# 03 - Degrade and augment (CCTV-style)

Turns the **undegraded** images in `data/synthetic_clean/{class}/` into the **degraded** training set
in `data/synthetic_degraded/{class}/`, simulating real restaurant-camera footage (low resolution,
noise, blur, lighting, JPEG artifacts, perspective). This is **novelty #1**.

Adapted from Avital's notebook to our pipeline: paths + classes come from `utils.py`, and the
**`unclassified`** class gets a *much heavier* corruption (until the plate state is unreadable) -
that heavy corruption is what *defines* that class (see `Project_Plan` 4.4 / CLAUDE.md).

Keep both copies on disk (clean = undegraded, degraded) so the with/without-degradation ablation
in step 05 can run.

## Imports + paths

In [ ]:
import os, random, shutil
import numpy as np
from PIL import Image, ImageFilter, ImageEnhance
import cv2

import sys
from pathlib import Path
# Make the local shlomi/utils.py importable whether launched from shlomi/ or the repo root.
for _cand in (Path.cwd(), Path.cwd() / "shlomi"):
    if (_cand / "utils.py").exists() and str(_cand) not in sys.path:
        sys.path.insert(0, str(_cand)); break
import utils

random.seed(utils.SEED); np.random.seed(utils.SEED)

INPUT_DIR  = utils.CLEAN_DIR          # data/synthetic_clean
OUTPUT_DIR = utils.DEGRADED_DIR       # data/synthetic_degraded

# Only the classes actually present on disk (a subset of the 5).
CLASSES = [cls for cls in utils.CLASS_NAMES if (INPUT_DIR / cls).is_dir()]
print("input :", INPUT_DIR)
print("output:", OUTPUT_DIR)
print("classes:", CLASSES)

def reset_degraded_folder():
    if OUTPUT_DIR.exists():
        print("Deleting existing degraded folder..."); shutil.rmtree(OUTPUT_DIR)
    for cls in CLASSES:
        os.makedirs(OUTPUT_DIR / cls, exist_ok=True)
    print("Fresh degraded folder ready:", OUTPUT_DIR)

## Degradation primitives

In [ ]:
def add_blur(img, radius_range=(0.3, 1.0)):
    return img.filter(ImageFilter.GaussianBlur(random.uniform(*radius_range)))

def add_noise(img, std_range=(10, 20)):
    arr = np.array(img).astype(np.float32)
    arr += np.random.normal(0, random.uniform(*std_range), arr.shape)
    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

def change_lighting(img, brightness_range=(0.5, 1.6), contrast_range=(0.5, 1.6)):
    img = ImageEnhance.Brightness(img).enhance(random.uniform(*brightness_range))
    img = ImageEnhance.Contrast(img).enhance(random.uniform(*contrast_range))
    return img

def downscale(img, sizes=(96, 128, 160, 192)):
    size = random.choice(sizes)
    return img.resize((size, size), Image.BILINEAR).resize((224, 224), Image.BILINEAR)

def compress(img, quality_range=(10, 50)):
    img.save("temp.jpg", quality=random.randint(*quality_range))
    return Image.open("temp.jpg")

def color_shift(img, shift_range=(0.7, 1.3)):
    arr = np.array(img).astype(np.float32)
    for ch in range(3):
        arr[:, :, ch] *= random.uniform(*shift_range)
    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

def rotate_image(img):
    return img.rotate(random.choice([0, 90, 180, 270]), expand=True)

def slight_rotate(img, angle_range=(-20, 20)):
    return img.rotate(random.uniform(*angle_range), resample=Image.BILINEAR, fillcolor=(0, 0, 0))

def perspective_transform(img, scale_range=(0.05, 0.15)):
    w, h = img.size
    shift = random.uniform(*scale_range) * w
    src = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
    dst = np.float32([
        [random.uniform(0, shift), random.uniform(0, shift)],
        [w - random.uniform(0, shift), random.uniform(0, shift)],
        [w - random.uniform(0, shift), h - random.uniform(0, shift)],
        [random.uniform(0, shift), h - random.uniform(0, shift)]])
    warped = cv2.warpPerspective(np.array(img), cv2.getPerspectiveTransform(src, dst),
                                 (w, h), borderMode=cv2.BORDER_REFLECT)
    return Image.fromarray(warped)

## The degradation pipeline

Per-image randomized. `unclassified` (`heavy=True`) gets far more aggressive downscale + blur +
noise + JPEG so the plate state becomes unreadable - that is what defines the class.

In [ ]:
def degrade_image(img, cls):
    heavy = (cls == utils.UNCLASSIFIED)     # corrupt until unreadable
    severity = random.random()

    # Resolution - the dominant CCTV cue. unclassified goes much lower-res.
    if heavy:
        size = random.choice([24, 32, 40, 56])
        img = img.resize((size, size), Image.BILINEAR).resize((224, 224), Image.BILINEAR)
    else:
        img = downscale(img)

    # Geometry
    if random.random() < 0.7:
        img = slight_rotate(img, (-10, 10) if severity < 0.5 else (-25, 25))
    if random.random() < 0.7:
        img = rotate_image(img)
    if random.random() < (0.8 if heavy else 0.6):
        img = perspective_transform(img, (0.12, 0.30) if heavy else
                                    ((0.05, 0.12) if severity < 0.5 else (0.12, 0.25)))

    # Blur (stronger for unclassified)
    if random.random() < (0.9 if heavy else 0.6):
        img = add_blur(img, (2.0, 4.0) if heavy else
                       ((0.3, 1.0) if severity < 0.5 else (1.0, 2.5)))

    # Noise (stronger for unclassified; gentlest for clean)
    if random.random() < (0.9 if heavy else 0.7):
        if heavy:        img = add_noise(img, (20, 45))
        elif cls == "clean": img = add_noise(img, (2, 8))
        else:            img = add_noise(img, (5, 15))

    # Lighting
    if random.random() < 0.6:
        rng = (0.6, 1.5) if heavy else (0.7, 1.4)
        img = change_lighting(img, rng, rng)

    # Color
    if random.random() < 0.5:
        img = color_shift(img, (0.85, 1.15))

    # JPEG compression (much lower quality for unclassified)
    if random.random() < (0.9 if heavy else 0.5):
        img = compress(img, (5, 20) if heavy else (30, 70))

    return img.convert("RGB")

## Generate the degraded dataset

2-5 augmentations per source image (more for `unclassified`).

In [ ]:
from tqdm import tqdm

def generate_augmented_dataset():
    reset_degraded_folder()
    for cls in CLASSES:
        in_dir, out_dir = INPUT_DIR / cls, OUTPUT_DIR / cls
        images = [p for p in in_dir.glob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png")]
        print(f"\n{cls}: {len(images)} source images")
        n_choices = [3, 4, 5, 6] if cls == utils.UNCLASSIFIED else [2, 3, 4, 5]
        for img_path in tqdm(images):
            img = Image.open(img_path).convert("RGB")
            for i in range(random.choice(n_choices)):
                random.seed(np.random.randint(0, 100000))
                degrade_image(img.copy(), cls).save(out_dir / f"{img_path.stem}_aug{i}.jpg", quality=95)

generate_augmented_dataset()
print("\nDONE.")
for cls in CLASSES:
    print(cls, len(list((OUTPUT_DIR / cls).glob('*'))))

## Balance classes (upsample to the largest)

In [ ]:
def upsample_to_max():
    sizes = {cls: len(list((OUTPUT_DIR / cls).glob('*'))) for cls in CLASSES}
    target = max(sizes.values())
    print("before:", sizes, "| target:", target)
    for cls in CLASSES:
        files = list((OUTPUT_DIR / cls).glob('*'))
        idx = 0
        while len(files) < target:
            src = random.choice(files)
            new = (OUTPUT_DIR / cls / f"{src.stem}_extra{idx}.jpg")
            degrade_image(Image.open(src).convert("RGB"), cls).save(new, quality=95)
            files.append(new); idx += 1
    print("after :", {cls: len(list((OUTPUT_DIR / cls).glob('*'))) for cls in CLASSES})

upsample_to_max()

## Visual check: clean (top) vs degraded (bottom) per class

In [ ]:
import matplotlib.pyplot as plt
n = 8
fig, axes = plt.subplots(2 * len(CLASSES), n, figsize=(2 * n, 2 * 2 * len(CLASSES)))
for r, cls in enumerate(CLASSES):
    clean_files = list((INPUT_DIR / cls).glob('*'))[:n]
    for i, cp in enumerate(clean_files):
        axes[2*r, i].imshow(Image.open(cp)); axes[2*r, i].axis('off')
        augs = list((OUTPUT_DIR / cls).glob(f"{cp.stem}_aug*.jpg"))
        ax = axes[2*r+1, i]
        ax.imshow(Image.open(random.choice(augs)) if augs else Image.open(cp)); ax.axis('off')
    axes[2*r, 0].set_ylabel(cls, fontsize=12)
plt.tight_layout(); plt.show()